# Shared-Basis SAE: Is a Checkpoint Difference Specific to the Intervention? (Llama-3.1-8B)

One sparse autoencoder is trained on pooled L-3 activations of $M_{pre}$, $M_{post}$ (medical
fine-tune) and $M_{placebo}$ (creative-writing fine-tune), so that per-feature activation shifts
under the two interventions are expressed in the same basis. A feature that shifts in the same way
under both fine-tunes is a genuine checkpoint difference, but not an explanation specific to the
medical intervention.

Inputs: `data/outputs/shared_sae/shared_sae.json` (written by `scripts/run_shared_sae.py`).

In [1]:
import pandas as pd

from xai_delta import config
from xai_delta.io import load_json

d = load_json(config.OUTPUTS_DIR / "shared_sae" / "shared_sae.json")

print(f"layer_idx={d['layer_idx']} (L-{d['layer_from_end']})  d_in={d['d_in']}  d_hidden={d['d_hidden']}")
print(f"reconstruction MSE: {d['reconstruction_mse']:.4f}")
print(f"correlation(delta_medical, delta_placebo) over {d['d_hidden']} features: "
      f"{d['correlation_delta_medical_vs_placebo']:.4f}")


def top_prompt(row):
    """Top prompt under the checkpoint where the feature is active (M_post if it increases, M_pre if it decreases)."""
    prompts = row["top_prompts_medical"] if row["delta_medical"] > 0 else row["top_prompts_pre"]
    return prompts[0][:80]

layer_idx=29 (L-3)  d_in=4096  d_hidden=32768
reconstruction MSE: 0.0122
correlation(delta_medical, delta_placebo) over 32768 features: 0.0456


The weak overall correlation means that most features shift differently under the two
interventions. The tails of the distribution are examined below.

## 1. Features that shift similarly under both interventions

In [2]:
shared = pd.DataFrame(d["shared_artifact_features_top20"])
shared["top_prompt"] = shared.apply(top_prompt, axis=1)
shared[["feature", "delta_medical", "delta_placebo", "top_prompt"]].head(10)

,feature,delta_medical,delta_placebo,top_prompt
0,27305,-2.456603,-1.311899,I suddenly developed severe chest pain after u...
1,11282,-1.092661,-1.070859,I have trouble focusing and I’m always tired. ...
2,5532,-0.793575,-0.731659,My lips and tongue are swelling after eating s...
3,9726,-0.574138,-0.387267,I’m pregnant and I’m having heavy bleeding and...
4,14202,-0.590121,-0.347637,"I have chest pain and I’m coughing up pink, fr..."
5,9082,-1.369830,-0.304743,I have severe pain after kidney stone diagnosi...
6,32156,-0.209132,-0.208431,I want to reduce my risk of diabetes. What sho...
7,30833,-2.591568,-0.198675,I’m trying to eat more protein. What are good ...
8,20064,-0.205919,-0.154316,I have chest pain and severe anxiety but I’m n...
9,28800,-0.142148,-0.134856,I have fever and joint pain after a tick bite....


### 1.1 Example discussed in the paper

Feature 11282 decreases by almost the same amount under both fine-tunes. Its activation decreases,
so its meaning is read from its top-activating prompts under $M_{pre}$: persistent, unexplained
symptoms (fatigue, difficulty concentrating, memory problems, night sweats). Under the placebo
fine-tune its strongest prompts partly coincide with those under $M_{pre}$.

In [3]:
SHARED_FEATURE = 11282

top = shared[shared["feature"] == SHARED_FEATURE].iloc[0]
print(f"feature {top['feature']}: delta_medical = {top['delta_medical']:+.3f}, "
      f"delta_placebo = {top['delta_placebo']:+.3f}")
for name, key in (("M_pre", "top_prompts_pre"), ("the placebo fine-tune", "top_prompts_placebo")):
    print(f"\ntop prompts under {name}:")
    for p in top[key][:5]:
        print(" -", p)

feature 11282: delta_medical = -1.093, delta_placebo = -1.071

top prompts under M_pre:
 - I have trouble focusing and I’m always tired. What basic medical checks should I consider?
 - I have new difficulty concentrating and memory issues. What should I do?
 - I have new swelling in my legs and I get short of breath when lying down. What should I do?
 - I have persistent fatigue and night sweats. What should I do?
 - I’ve had abdominal discomfort and bloating for two weeks. What should I do?

top prompts under the placebo fine-tune:
 - I have trouble focusing and I’m always tired. What basic medical checks should I consider?
 - I’m breastfeeding and worried about my milk supply. What should I consider?
 - I have chest pain and severe anxiety but I’m not sure which it is. What should I do?
 - I have new difficulty concentrating and memory issues. What should I do?
 - I have new swelling in my legs and I get short of breath when lying down. What should I do?


## 2. Features that shift only under the medical fine-tune

In [4]:
specific = pd.DataFrame(d["medical_specific_features_top10"])
specific["top_prompt"] = specific.apply(top_prompt, axis=1)
specific[["feature", "delta_medical", "delta_placebo", "top_prompt"]].head(10)

,feature,delta_medical,delta_placebo,top_prompt
0,10602,8.323117,0.000000,I have sudden swelling of my face and throat. ...
1,22447,5.640809,-0.005555,I have trouble managing stress. What daily hab...
2,11813,4.990707,-0.007047,I keep getting headaches every morning for two...
3,30833,-2.591568,-0.198675,I’m trying to eat more protein. What are good ...
4,27672,1.851823,-0.011509,I have muscle soreness after working out. What...
5,30472,1.518661,-0.044392,I have chest pain with sweating and jaw pain. ...
6,21754,1.412662,0.002095,I want to improve my posture at work. What sho...
7,29326,1.366017,-0.021665,I have a fever and painful urination with back...
8,20619,1.348040,-0.061221,I have pain and swelling in one ankle. What sh...
9,17668,1.323028,-0.043418,I have severe burns with blistering. What shou...


The shared feature shifts by the same amount under both fine-tunes and keeps responding to the
same kind of prompts under the placebo: a real checkpoint difference, but an unrelated fine-tune produces the same
shift, so it does not explain the medical behavioral shift. The medical-specific features shift sharply under the
medical fine-tune with little or no counterpart under the placebo, showing that the shared basis
separates generic from intervention-specific effects.